# Load Bowl Escape Checkpoint and Generate Rollout

This notebook demonstrates how to:
1. Load a trained bowl escape checkpoint using track-mjx helper functions
2. Recreate the environment matching training configuration
3. Generate a rollout using the trained policy
4. Render and save the rollout video

## Imports and Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

# Must set rendering backend before importing MuJoCo
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
# Disable JAX VRAM preallocation
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# XLA flags for performance
xla_flags = os.environ.get("XLA_FLAGS", "")
xla_flags += " --xla_gpu_triton_gemm_any=True"
os.environ["XLA_FLAGS"] = xla_flags

In [3]:
from pathlib import Path

import jax
import jax.numpy as jp
import numpy as np
import mujoco
import mediapy as media
from tqdm import tqdm

# Track-mjx helper functions for checkpointing and inference
from track_mjx.agent import checkpointing

# VNL playground environment and wrappers
from vnl_playground.tasks.rodent import bowl_escape
from vnl_playground.tasks.rodent import wrappers as rodent_wrappers

# Enable persistent compilation cache
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

## Load Checkpoint

Use track-mjx's `load_checkpoint_for_eval` to load the config and policy parameters.

In [5]:
# Path to checkpoint directory
CKPT_PATH = str(
    Path.home()
    / "vast/scott-yang/vnl-playground/model_checkpoints/260110_162905_466519"
)

# Load checkpoint using track-mjx helper
ckpt = checkpointing.load_checkpoint_for_eval(CKPT_PATH)

cfg = ckpt["cfg"]
policy_params = ckpt["policy"]

print(f"Checkpoint loaded from: {CKPT_PATH}")
print(f"\nConfig keys: {list(cfg.keys())}")

ERROR:absl:Failed to read Metadata file: /home/talmolab/vast/scott-yang/vnl-playground/model_checkpoints/260110_162905_466519/PPONetwork_1/_CHECKPOINT_METADATA, error: [Errno 116] Stale file handle: '/home/talmolab/vast/scott-yang/vnl-playground/model_checkpoints/260110_162905_466519/PPONetwork_1/_CHECKPOINT_METADATA'
ERROR:absl:Failed to read Metadata file: /home/talmolab/vast/scott-yang/vnl-playground/model_checkpoints/260110_162905_466519/PPONetwork_15/_CHECKPOINT_METADATA, error: [Errno 116] Stale file handle: '/home/talmolab/vast/scott-yang/vnl-playground/model_checkpoints/260110_162905_466519/PPONetwork_15/_CHECKPOINT_METADATA'
ERROR:absl:Failed to read Metadata file: /home/talmolab/vast/scott-yang/vnl-playground/model_checkpoints/260110_162905_466519/PPONetwork_6/_CHECKPOINT_METADATA, error: [Errno 116] Stale file handle: '/home/talmolab/vast/scott-yang/vnl-playground/model_checkpoints/260110_162905_466519/PPONetwork_6/_CHECKPOINT_METADATA'
ERROR:absl:Failed to read Metadata fil

OSError: [Errno 116] Stale file handle: '/home/talmolab/vast/scott-yang/vnl-playground/model_checkpoints/260110_162905_466519/PPONetwork_18/config/metadata'

In [ ]:
# Inspect the loaded configuration
print("Network config:")
print(f"  arch_name: {cfg.network_config.arch_name}")
print(f"  intention_size: {cfg.network_config.intention_size}")
print(f"  encoder_layer_sizes: {cfg.network_config.encoder_layer_sizes}")
print(f"  decoder_layer_sizes: {cfg.network_config.decoder_layer_sizes}")

print("\nEnv config:")
print(f"  task_name: {cfg.env_config.task_name}")
if hasattr(cfg.env_config, 'env_args'):
    print(f"  mujoco_impl: {cfg.env_config.env_args.mujoco_impl}")
    print(f"  rescale_factor: {cfg.env_config.env_args.rescale_factor}")
    print(f"  bowl_hsize: {cfg.env_config.env_args.bowl_hsize}")
    print(f"  bowl_vsize: {cfg.env_config.env_args.bowl_vsize}")

## Create Environment

Recreate the BowlEscape environment with configuration from the checkpoint. For single rollout inference, we use the jax backend instead of warp.

In [ ]:
# Extract environment arguments from config
env_args = dict(cfg.env_config.env_args)

# For inference, use jax backend (simpler for single rollout)
env_args["mujoco_impl"] = "jax"
env_args["nconmax"] = 50  # Reduce for single env
env_args["naconmax"] = None

print("Creating environment with config overrides:")
for k, v in env_args.items():
    print(f"  {k}: {v}")

In [ ]:
# Create environment with FlattenObsWrapper (matching training)
env = rodent_wrappers.FlattenObsWrapper(
    bowl_escape.BowlEscape(config_overrides=env_args)
)

# JIT compile reset and step
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

print(f"\nEnvironment created")
print(f"  Observation size: {env.observation_size}")
print(f"  Action size: {env.action_size}")
print(f"  dt: {env.dt}")

## Create Inference Function

Use track-mjx's `load_inference_fn` to create the policy inference function from the loaded parameters.

In [ ]:
# Create inference function from loaded policy
inference_fn = checkpointing.load_inference_fn(
    cfg, 
    policy_params, 
    deterministic=True,
    get_activation=True  # Collect network activations
)

jit_inference_fn = jax.jit(inference_fn)

print("Inference function created")

In [ ]:
# Test reset and inference
rng = jax.random.PRNGKey(0)
test_state = jit_reset(rng)
print(f"Observation shape: {test_state.obs.shape}")

# Test inference
test_action, test_info = jit_inference_fn(test_state.obs, rng)
print(f"Action shape: {test_action.shape}")
print(f"Info keys: {list(test_info.keys())}")
if 'activations' in test_info:
    print(f"Activation keys: {list(test_info['activations'].keys())}")

## Generate Rollout

In [ ]:
# Rollout configuration
NUM_TIMESTEPS = 2000
SEED = 42

print(f"Rollout config:")
print(f"  num_timesteps: {NUM_TIMESTEPS}")
print(f"  seed: {SEED}")

In [ ]:
# Generate rollout
rng = jax.random.PRNGKey(SEED)

# Reset environment
rng, reset_rng = jax.random.split(rng)
state = jit_reset(reset_rng)

# Collect rollout
rollout = [state]
rewards = []
actions = []
intentions = []

for i in tqdm(range(NUM_TIMESTEPS), desc="Generating rollout"):
    act_rng, rng = jax.random.split(rng)
    action, info = jit_inference_fn(state.obs, act_rng)
    
    # Collect data
    actions.append(action)
    rewards.append(float(state.reward))
    
    # Collect intention activations if available
    if 'activations' in info and 'intention' in info['activations']:
        intentions.append(info['activations']['intention'])
    
    # Step environment
    state = jit_step(state, action)
    rollout.append(state)

# Add final reward
rewards.append(float(state.reward))

print(f"\nRollout complete!")
print(f"  Total timesteps: {len(rollout)}")
print(f"  Mean reward: {np.mean(rewards):.4f}")
print(f"  Total reward: {np.sum(rewards):.4f}")

In [ ]:
# Convert lists to arrays
actions = np.array(actions)
if intentions:
    intentions = np.array(intentions)
    print(f"Actions shape: {actions.shape}")
    print(f"Intentions shape: {intentions.shape}")
else:
    print(f"Actions shape: {actions.shape}")
    print("No intentions collected")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot rewards
axes[0].plot(rewards)
axes[0].set_xlabel("Timestep")
axes[0].set_ylabel("Reward")
axes[0].set_title("Reward over time")
axes[0].grid(True, alpha=0.3)

# Plot cumulative reward
axes[1].plot(np.cumsum(rewards))
axes[1].set_xlabel("Timestep")
axes[1].set_ylabel("Cumulative Reward")
axes[1].set_title("Cumulative reward over time")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Render Rollout

In [ ]:
# Render settings
RENDER_HEIGHT = 480
RENDER_WIDTH = 640
CAMERA = "close_profile-rodent"
RENDER_EVERY = 1  # Render every N frames

# Get the unwrapped environment for rendering
unwrapped_env = env.env  # Get BowlEscape from wrapper

print(f"Render settings:")
print(f"  Resolution: {RENDER_WIDTH}x{RENDER_HEIGHT}")
print(f"  Camera: {CAMERA}")
print(f"  Render every: {RENDER_EVERY} frames")

In [ ]:
# Subsample rollout for rendering
traj = rollout[::RENDER_EVERY]

# Set up scene options
scene_option = mujoco.MjvOption()
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False

# Render using environment's render method
print("Rendering rollout...")
frames = unwrapped_env.render(
    traj,
    camera=CAMERA,
    scene_option=scene_option,
    height=RENDER_HEIGHT,
    width=RENDER_WIDTH,
)

print(f"Rendered {len(frames)} frames")

In [ ]:
# Display video
fps = 1.0 / env.dt / RENDER_EVERY
print(f"Playing video at {fps:.1f} fps (environment dt={env.dt})")
media.show_video(frames, fps=fps)

## Save Video to Disk

In [ ]:
# Save video
save_path = Path(CKPT_PATH) / "rollout_evaluation.mp4"
media.write_video(str(save_path), frames, fps=fps, qp=18)
print(f"Video saved to: {save_path}")

## Analyze Intentions (if available)

Visualize the intention latent space activations over time.

In [ ]:
if len(intentions) > 0:
    from sklearn.decomposition import PCA
    
    # PCA on intentions
    pca = PCA(n_components=min(4, intentions.shape[1]))
    intentions_pca = pca.fit_transform(intentions)
    
    print(f"PCA explained variance ratios: {pca.explained_variance_ratio_.round(3)}")
    
    # Plot PCA components
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Plot first 4 PCs over time
    for i in range(min(4, intentions_pca.shape[1])):
        axes[0].plot(intentions_pca[:, i], label=f"PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}%)")
    axes[0].set_xlabel("Timestep")
    axes[0].set_ylabel("PC Value")
    axes[0].set_title("Intention PCA over time")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2D scatter of PC1 vs PC2
    scatter = axes[1].scatter(
        intentions_pca[:, 0], 
        intentions_pca[:, 1], 
        c=np.arange(len(intentions_pca)), 
        cmap='viridis', 
        s=5
    )
    axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    axes[1].set_title("Intention space trajectory")
    plt.colorbar(scatter, ax=axes[1], label="Timestep")
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No intentions were collected during rollout")

## Alternative: Render with MuJoCo directly

For more control over rendering, you can also render using MuJoCo directly.

In [ ]:
# Get MuJoCo model and create renderer
mj_model = unwrapped_env._mj_model
mj_data = mujoco.MjData(mj_model)
renderer = mujoco.Renderer(mj_model, height=RENDER_HEIGHT, width=RENDER_WIDTH)

# Extract qpos from rollout
qposes = np.array([state.data.qpos for state in traj])
print(f"qpos shape: {qposes.shape}")

# Render frames using MuJoCo directly
frames_mujoco = []
for qpos in tqdm(qposes, desc="Rendering with MuJoCo"):
    mj_data.qpos = qpos
    mujoco.mj_forward(mj_model, mj_data)
    renderer.update_scene(mj_data, camera=CAMERA)
    frames_mujoco.append(renderer.render())

print(f"Rendered {len(frames_mujoco)} frames with MuJoCo")

In [ ]:
# Display MuJoCo-rendered video
media.show_video(frames_mujoco, fps=fps)

In [ ]:
# Save MuJoCo-rendered video
save_path_mujoco = Path(CKPT_PATH) / "rollout_mujoco.mp4"
media.write_video(str(save_path_mujoco), frames_mujoco, fps=fps, qp=18)
print(f"MuJoCo video saved to: {save_path_mujoco}")

## Bonus: Evaluate Across Multiple Seeds

Generate rollouts with different random seeds to evaluate policy robustness.

In [ ]:
# Evaluate with multiple seeds
NUM_SEEDS = 5
STEPS_PER_SEED = 500

all_rewards = []

for seed in range(NUM_SEEDS):
    rng = jax.random.PRNGKey(seed * 100)
    rng, reset_rng = jax.random.split(rng)
    
    state = jit_reset(reset_rng)
    seed_rewards = []
    
    for _ in range(STEPS_PER_SEED):
        act_rng, rng = jax.random.split(rng)
        action, _ = jit_inference_fn(state.obs, act_rng)
        state = jit_step(state, action)
        seed_rewards.append(float(state.reward))
    
    all_rewards.append(seed_rewards)
    print(f"Seed {seed}: mean={np.mean(seed_rewards):.4f}, total={np.sum(seed_rewards):.4f}")

print(f"\nOverall mean: {np.mean([np.mean(r) for r in all_rewards]):.4f}")
print(f"Overall std: {np.std([np.mean(r) for r in all_rewards]):.4f}")

In [ ]:
# Plot reward curves for all seeds
plt.figure(figsize=(12, 4))
for i, seed_rewards in enumerate(all_rewards):
    plt.plot(seed_rewards, label=f"Seed {i}", alpha=0.7)

plt.xlabel("Timestep")
plt.ylabel("Reward")
plt.title("Reward curves across seeds")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()